# Archivo exploratorio / Exploratory archive

El flujo vigente está en REPRODUCIBILITY.md y los resultados en reports/Executive_Report.pdf. Este notebook conserva exploración previa y no prueba políticas de inventario ni ahorro operativo. / Follow the reviewed reproducibility guide for current results; this notebook is not evidence of an operational deployment.

Revisión metodológica / Method review: ver ../DECISION_REVIEW.md. Cierres hasta 2035; cohortes no equivalen a ventas de dos años. Compras no equivalen a demanda. Ahorros y políticas requieren validación.

# Market Stall — SQL Analysis (DuckDB)

**Portfolio note.** Same project as `Market_Stall_Analysis.ipynb`, adaptation/translation of an original
Spanish project. All supplier names/contacts/locations are fictitious; amounts are approximate (1 USD =
935 CLP). See that notebook for the full narrative — this one exists to show the **same KPIs written as
SQL**: joins, CTEs (`WITH`), and window functions (`RANK`, running `SUM() OVER`, etc.), since that's the
core skill most Data Analyst postings ask for and the rest of this project is Python/Excel only.

**Engine:** [DuckDB](https://duckdb.org) — an embedded, file-free SQL engine that queries CSVs (and
Parquet, Pandas DataFrames) directly with full standard SQL, including window functions and CTEs. No
server, no separate database file to manage — perfect for a portfolio piece that has to just run.

**Sections**
1. Setup & load tables
2. Basic exploration (row counts, joins)
3. Revenue, margin & waste by category
4. Top-N products by margin (`RANK()` window function)
5. ABC / Pareto classification (running `SUM() OVER`)
6. Monthly revenue trend (`DATE_TRUNC`)
7. Supplier scorecard & HHI concentration (CTEs)
8. Backup vs. primary supplier premium (self-join)
9. Restock alerts (multi-way join + `CASE`)


In [1]:
import sys, subprocess
def ensure(pkgs):
    for p in pkgs:
        mod = p.split("==")[0]
        try:
            __import__(mod)
        except ImportError:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", p])
ensure(["duckdb", "pandas"])
print("Libraries ready.")


Libraries ready.


## 1. Setup & load tables

In [2]:
import duckdb
import pandas as pd
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 150)

con = duckdb.connect(database=":memory:")

# Load the source CSVs straight into DuckDB tables — no separate ETL step needed.
con.execute("CREATE TABLE products AS SELECT * FROM read_csv_auto('products.csv')")
con.execute("CREATE TABLE suppliers AS SELECT * FROM read_csv_auto('suppliers.csv')")
con.execute("CREATE TABLE supply_links AS SELECT * FROM read_csv_auto('supply_links.csv')")
con.execute("CREATE TABLE purchases AS SELECT * FROM read_csv_auto('purchases.csv')")
con.execute("CREATE TABLE batch_outcomes AS SELECT * FROM read_csv_auto('batch_outcomes.csv')")

for t in ["products","suppliers","supply_links","purchases","batch_outcomes"]:
    n = con.sql(f"SELECT COUNT(*) AS n FROM {t}").df()["n"][0]
    print(f"{t:15s} {n:>6} rows")


products           128 rows
suppliers           22 rows
supply_links       256 rows
purchases         3825 rows
batch_outcomes    3825 rows


## 2. Basic exploration

In [3]:
con.sql('''
    SELECT Category, COUNT(*) AS n_products,
           ROUND(AVG(SalePrice_CLP),0) AS avg_price_clp,
           SUM(CASE WHEN Essential='Yes' THEN 1 ELSE 0 END) AS n_essential
    FROM products
    GROUP BY Category
    ORDER BY n_products DESC
''').df()


,Category,n_products,avg_price_clp,n_essential
0,Candy,17,1088.0,0.0
1,Packaging,17,3059.0,0.0
2,Nuts,15,17367.0,1.0
3,Dried Fruit,12,6458.0,2.0
4,Olives,11,3227.0,4.0
5,Seasonings,11,1500.0,2.0
6,Chili,9,1889.0,2.0
7,Peanuts,9,7944.0,1.0
8,Pickles,5,2800.0,0.0
9,Honey,5,5500.0,1.0


## 3. Revenue, margin & waste by category

A single query joining **purchases** (what was bought) to **batch_outcomes** (how it was sold/wasted),
then to **products** for category — the join pattern any of these KPIs need.


In [4]:
revenue_by_cat = con.sql('''
    WITH batch_full AS (
        SELECT p.ProductID, p.Category, pr.SalePrice_USD,
               o.QtyPurchased, o.QtySold, o.QtyWasted, o.Revenue_USD, o.GrossProfit_USD
        FROM batch_outcomes o
        JOIN purchases p   ON p.BatchID = o.BatchID
        JOIN products pr   ON pr.ProductID = p.ProductID
    )
    SELECT Category,
           ROUND(SUM(Revenue_USD),0)      AS revenue_usd,
           ROUND(SUM(GrossProfit_USD),0)  AS gross_profit_usd,
           ROUND(100.0*SUM(QtyWasted)/SUM(QtyPurchased), 1) AS waste_pct
    FROM batch_full
    GROUP BY Category
    ORDER BY revenue_usd DESC
''').df()
revenue_by_cat


,Category,revenue_usd,gross_profit_usd,waste_pct
0,Nuts,136803.0,31488.0,1.9
1,Olives,79921.0,19493.0,5.2
2,Peanuts,58004.0,15207.0,2.8
3,Dried Fruit,31198.0,8184.0,1.5
4,Grains,24363.0,4436.0,9.2
5,Chili,18513.0,5312.0,0.7
6,Trail Mix,17460.0,4143.0,3.1
7,Candy,16854.0,5060.0,1.4
8,Packaging,14920.0,4910.0,0.0
9,Seasonings,11448.0,3476.0,0.4


## 4. Top products by margin — `RANK()` window function

Window functions compute a value across a set of rows *without collapsing them into one row per group* —
here, ranking every product by margin while still returning one row per product.


In [5]:
con.sql('''
    WITH margins AS (
        SELECT pr.ProductID, pr.Product, pr.Category, pr.SalePrice_CLP,
               sl.PurchasePrice_CLP,
               ROUND(100.0*(pr.SalePrice_CLP - sl.PurchasePrice_CLP)/pr.SalePrice_CLP, 1) AS margin_pct
        FROM products pr
        JOIN supply_links sl ON sl.ProductID = pr.ProductID AND sl.SupplierRole = 'Primary'
    )
    SELECT Product, Category, margin_pct,
           RANK() OVER (ORDER BY margin_pct DESC) AS margin_rank
    FROM margins
    QUALIFY margin_rank <= 10
    ORDER BY margin_rank
''').df()


,Product,Category,margin_pct,margin_rank
0,Poly bag 12x20cm (pack of 100),Packaging,36.7,1
1,Poly bag 30x40cm (pack of 100),Packaging,36.3,2
2,Licorice bag 100g,Candy,35.0,3
3,"Bay leaves, bag 50g",Seasonings,35.0,3
4,Poly bag 10x15cm (pack of 100),Packaging,35.0,3
5,Coconut candy bag 100g,Candy,35.0,3
6,Gummy candy bag 100g,Candy,35.0,3
7,Poly bag 15x25cm (pack of 100),Packaging,35.0,3
8,Poly bag 25x35cm (pack of 100),Packaging,35.0,3
9,Toffee caramels bag 100g,Candy,35.0,3


## 5. ABC classification — running total with `SUM() OVER`

The classic Pareto pattern in SQL: a running cumulative sum ordered by revenue, divided by the grand
total, all computed in the `SELECT` with no self-join and no subquery per row.


In [6]:
abc = con.sql('''
    WITH rev AS (
        SELECT p.ProductID, pr.Product, pr.Category, SUM(o.Revenue_USD) AS revenue_usd
        FROM batch_outcomes o
        JOIN purchases p ON p.BatchID = o.BatchID
        JOIN products pr ON pr.ProductID = p.ProductID
        GROUP BY p.ProductID, pr.Product, pr.Category
    ),
    ranked AS (
        SELECT *,
               SUM(revenue_usd) OVER (ORDER BY revenue_usd DESC
                                       ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_total,
               SUM(revenue_usd) OVER () AS grand_total
        FROM rev
    )
    SELECT ProductID, Product, Category, ROUND(revenue_usd,0) AS revenue_usd,
           ROUND(100.0*running_total/grand_total, 1) AS cum_pct,
           CASE WHEN 100.0*running_total/grand_total <= 80 THEN 'A'
                WHEN 100.0*running_total/grand_total <= 95 THEN 'B'
                ELSE 'C' END AS abc_class
    FROM ranked
    ORDER BY revenue_usd DESC
''').df()

print(abc["abc_class"].value_counts().to_string())
abc.head(8)


abc_class
A    56
B    43
C    29


,ProductID,Product,Category,revenue_usd,cum_pct,abc_class
0,P065,"Pistachios (roasted, salted)",Nuts,16676.0,3.8,A
1,P066,Pistachios (unsalted),Nuts,16214.0,7.4,A
2,P063,Cashews (salted),Nuts,13876.0,10.6,A
3,P064,Cashews (unsalted),Nuts,13276.0,13.6,A
4,P069,Macadamia nuts,Nuts,12164.0,16.3,A
5,P062,European hazelnuts (premium),Nuts,11405.0,18.9,A
6,P067,Pecans (shelled),Nuts,9443.0,21.1,A
7,P051,Chocolate-coated peanuts,Peanuts,9216.0,23.2,A


## 6. Monthly revenue trend — `DATE_TRUNC`

In [7]:
con.sql('''
    SELECT DATE_TRUNC('month', CAST(p.PurchaseDate AS DATE)) AS month,
           ROUND(SUM(o.Revenue_USD),0) AS revenue_usd,
           ROUND(100.0*SUM(o.QtyWasted)/SUM(o.QtyPurchased), 1) AS waste_pct
    FROM batch_outcomes o
    JOIN purchases p ON p.BatchID = o.BatchID
    GROUP BY month
    ORDER BY month
''').df().head(12)


,month,revenue_usd,waste_pct
0,2023-07-01,19975.0,3.8
1,2023-08-01,19974.0,3.2
2,2023-09-01,22866.0,2.6
3,2023-10-01,16117.0,2.8
4,2023-11-01,15155.0,3.2
5,2023-12-01,18974.0,1.7
6,2024-01-01,15545.0,4.3
7,2024-02-01,14415.0,6.9
8,2024-03-01,15262.0,2.7
9,2024-04-01,19516.0,3.2


## 7. Supplier scorecard & HHI concentration — CTEs

Two CTEs chained together: one aggregates per-supplier stats, the second turns that into a market-share
squared sum (the Herfindahl-Hirschman Index) for the whole supplier base.


In [8]:
scorecard = con.sql('''
    WITH primary_links AS (
        SELECT sl.SupplierName, sl.ProductID, pr.Essential, sl.LeadTime_Days
        FROM supply_links sl
        JOIN products pr ON pr.ProductID = sl.ProductID
        WHERE sl.SupplierRole = 'Primary'
    ),
    per_supplier AS (
        SELECT SupplierName,
               COUNT(*) AS n_products,
               SUM(CASE WHEN Essential='Yes' THEN 1 ELSE 0 END) AS n_essential,
               ROUND(AVG(LeadTime_Days),1) AS avg_lead_time
        FROM primary_links
        GROUP BY SupplierName
    )
    SELECT *,
           ROUND(100.0*n_products / SUM(n_products) OVER (), 1) AS pct_of_products
    FROM per_supplier
    ORDER BY n_products DESC
''').df()
scorecard


,SupplierName,n_products,n_essential,avg_lead_time,pct_of_products
0,Maipo Nuts Ltd.,34,2.0,1.6,26.6
1,Molina Artisan Sweets,17,0.0,2.2,13.3
2,PackChile Wholesale,17,0.0,1.4,13.3
3,San Fernando Dried Goods,15,3.0,1.6,11.7
4,El Olivar Olives Ltd.,11,4.0,2.2,8.6
5,Valley Spices SpA,11,2.0,1.6,8.6
6,Curico Chili & Sauces,9,2.0,1.7,7.0
7,Doña Rosa Pickles,5,0.0,2.2,3.9
8,Los Ulmos Apiary,5,1.0,2.2,3.9
9,San Vicente Grains & Legumes,4,1.0,1.8,3.1


In [9]:
hhi = ((scorecard["pct_of_products"])**2).sum()
level = "low" if hhi<1500 else ("moderate" if hhi<2500 else "high")
print(f"Supplier HHI (via SQL-computed shares): {hhi:.0f} / 10000 -> {level} concentration")


Supplier HHI (via SQL-computed shares): 1435 / 10000 -> low concentration


## 8. Backup vs. primary supplier premium — self-join

A self-join on `supply_links`, matching each product's Primary row to its own Backup row, to price the
"cost of resilience" directly in SQL.


In [10]:
con.sql('''
    SELECT p.ProductID, prim.PurchasePrice_CLP AS primary_price,
           bak.PurchasePrice_CLP AS backup_price,
           ROUND(100.0*(bak.PurchasePrice_CLP - prim.PurchasePrice_CLP)/prim.PurchasePrice_CLP, 1) AS backup_premium_pct
    FROM products p
    JOIN supply_links prim ON prim.ProductID = p.ProductID AND prim.SupplierRole = 'Primary'
    JOIN supply_links bak  ON bak.ProductID  = p.ProductID AND bak.SupplierRole  = 'Backup'
    ORDER BY backup_premium_pct DESC
    LIMIT 10
''').df()


,ProductID,primary_price,backup_price,backup_premium_pct
0,P120,950,1100,15.8
1,P061,5400,6250,15.7
2,P080,650,750,15.4
3,P090,650,750,15.4
4,P088,650,750,15.4
5,P053,5900,6750,14.4
6,P013,1050,1200,14.3
7,P067,13000,14800,13.8
8,P110,9950,11300,13.6
9,P105,4300,4850,12.8


## 9. Restock alerts — multi-way join + `CASE`


In [11]:
con.sql('''
    SELECT pr.ProductID, pr.Product, pr.Category, pr.CurrentStock,
           sl.SupplierName AS primary_supplier, s.Phone,
           CASE WHEN pr.Essential='Yes' AND pr.CurrentStock='Low'    THEN 'URGENT'
                WHEN pr.Essential='Yes' AND pr.CurrentStock='Medium' THEN 'Watch'
                WHEN pr.Essential='Yes'                              THEN 'OK'
                ELSE 'Low priority' END AS reorder_priority
    FROM products pr
    JOIN supply_links sl ON sl.ProductID = pr.ProductID AND sl.SupplierRole = 'Primary'
    JOIN suppliers s     ON s.SupplierID = sl.SupplierID
    WHERE pr.Essential = 'Yes'
    ORDER BY CASE reorder_priority WHEN 'URGENT' THEN 1 WHEN 'Watch' THEN 2 ELSE 3 END
    LIMIT 15
''').df()


,ProductID,Product,Category,CurrentStock,primary_supplier,Phone,reorder_priority
0,P012,Cooked mote (hominy wheat),Grains,Low,San Vicente Grains & Legumes,+56 9 4422 1188,URGENT
1,P002,"Olives 140/160 (large), brined",Olives,Low,El Olivar Olives Ltd.,+56 9 5511 2233,URGENT
2,P021,"Ground golden chili (aji de color), bag 100g",Chili,Medium,Curico Chili & Sauces,+56 9 7788 1122,Watch
3,P092,Golden raisins,Raisins,Medium,San Fernando Dried Goods,+56 9 5566 7788,Watch
4,P003,"Olives 160/180, brined",Olives,Medium,El Olivar Olives Ltd.,+56 9 5511 2233,Watch
5,P095,Dried prunes (with pit),Dried Fruit,Medium,San Fernando Dried Goods,+56 9 5566 7788,Watch
6,P097,"Dried peaches (huesillos), large",Dried Fruit,Medium,San Fernando Dried Goods,+56 9 5566 7788,Watch
7,P031,"Ground cumin, bag 100g",Seasonings,Medium,Valley Spices SpA,+56 9 3344 5566,Watch
8,P046,Roasted salted peanuts,Peanuts,Medium,Maipo Nuts Ltd.,+56 9 2211 3344,Watch
9,P055,Walnuts (shelled),Nuts,Medium,Maipo Nuts Ltd.,+56 9 2211 3344,Watch


## Closing note

Every KPI in the Python/pandas notebook has a SQL equivalent above using only joins, `WITH` CTEs, `CASE`,
and window functions (`RANK`, `SUM() OVER`, `QUALIFY`) — no procedural loops. DuckDB queries the CSVs
directly, so this also doubles as a lightweight, file-free alternative to standing up a full database
just to demonstrate the SQL.
